<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [4]</a>'.</span>

# Automated Reasoning Evaluation

This notebook demonstrates how to evaluate **Automated Reasoning (AR) Checks** in Amazon Bedrock Guardrails.

AR Checks verify LLM outputs against formal policy rules using a two-step pipeline:
1. **Translation** (LLM-based): Natural language → logical formulas
2. **Verification** (SMT solver): Check logical formulas against policy rules

This produces one of **7 validation result types**, grouped by outcome:

**Definitive answers** — the solver reached a conclusion:

| Validation Result | Meaning | Actionable? |
|---------|---------|-------------|
| `VALID` | Claim satisfies all translated policy rules | Serve the response |
| `INVALID` | Claim violates at least one policy rule | Block or rewrite the response |
| `IMPOSSIBLE` | Contradictory premises detected | Review input for logical inconsistencies |

**Partial answers** — some signal, but not conclusive:

| Validation Result | Meaning | Actionable? |
|---------|---------|-------------|
| `SATISFIABLE` | Some interpretations are valid, some are not | Add constraints or ask for clarification |
| `TRANSLATION_AMBIGUOUS` | Multiple interpretations of the input | Improve variable descriptions or ask user to clarify |

**No answer** — the system couldn't fully evaluate the claim:

| Validation Result | Meaning | Actionable? |
|---------|---------|-------------|
| `NO_TRANSLATIONS` | No policy variables matched the input | Add variables/rules to cover the input, or rephrase the input to match existing policy terms |
| `TOO_COMPLEX` | SMT solver timed out | Simplify the claim or split into parts |

> **Time:** ~20 minutes (or ~5 minutes with an existing guardrail)
>
> **Deep dives available:**
> - `04-06-02` — Document preprocessing pipeline (standalone, ~30 min)
> - `04-06-03` — Comprehensive evaluation with full test suite (~30 min)
> - `04-06-04` — Multi-guardrail architecture + fidelity optimization (~45 min, requires MCP)

In [1]:
!uv venv .venv 2>/dev/null; source .venv/bin/activate && uv pip install -q -r requirements.txt

zsh:source:1: no such file or directory: .venv/bin/activate


In [2]:
import json
import os
import time
import re
import uuid
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from botocore.config import Config

# AR finding types — grouped by outcome, with definitive answers taking
# priority over partial answers, and partial over no-answer.
# Within each group, the order reflects which result is more actionable
# (e.g. INVALID is more urgent than IMPOSSIBLE).
FINDING_TYPES = [
    'translationAmbiguous', 'impossible', 'invalid',
    'satisfiable', 'valid', 'tooComplex', 'noTranslations'
]

# Priority order for aggregation: when multiple findings exist, pick the
# most actionable one. Lower index = higher priority.
FINDING_PRIORITY = {ft: i for i, ft in enumerate(FINDING_TYPES)}

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [3]:
config = Config(retries={'max_attempts': 3})
bedrock_client = boto3.client('bedrock', config=config)
bedrock_runtime_client = boto3.client('bedrock-runtime', config=config)

try:
    bedrock_client.list_guardrails(maxResults=1)
    print("Bedrock access verified")
except Exception as e:
    print(f"ERROR: Cannot access Bedrock. Check your AWS credentials.\n{e}")

Bedrock access verified


## Configuration: Choose Your Path

Pick one of three options:

- **Option A — Use an existing guardrail** (~1 minute): You already have a guardrail with AR policies attached. Provide the guardrail ID and skip straight to evaluation.
- **Option B — Bring your own AR policy ARNs** (~2 minutes): You have AR policy ARNs (e.g. from the Bedrock console or notebook `04-06-04`) but no guardrail yet. We'll create the guardrail for you.
- **Option C — Build everything from scratch** (~10-15 minutes): Create new AR policies from the housing code, build them, and create a guardrail.

> **Tip:** If you've already run this notebook, your previous guardrail ID is saved in `data/ar_eval_state.json`.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [4]:
# ── Pick your path: "A", "B", or "C" ────────────────────────
PATH = "A"

# ── Option A: Use an existing guardrail ─────────────────────
# Fill these in if PATH = "A":
EXISTING_GUARDRAIL_ID = "sthtufz0pzlh"       # e.g. "s9bf946q77mr"
EXISTING_GUARDRAIL_VERSION = "DRAFT"  # e.g. "DRAFT" or "1"

# ── Option B: Bring your own AR policy ARNs ─────────────────
# Fill these in if PATH = "B" — versioned ARNs (with :version suffix):
EXISTING_POLICY_ARNS = [
    # e.g. "arn:aws:bedrock:us-west-2:123456789012:automated-reasoning-policy/abc123:1",
    # e.g. "arn:aws:bedrock:us-west-2:123456789012:automated-reasoning-policy/def456:1",
]

# ── Validation ──────────────────────────────────────────────
PATH = PATH.upper()
assert PATH in ("A", "B", "C"), f"PATH must be 'A', 'B', or 'C', got '{PATH}'"

if PATH == "A":
    # Auto-load from previous run if no ID provided
    if not EXISTING_GUARDRAIL_ID:
        import os
        state_path = 'data/ar_eval_state.json'
        if os.path.exists(state_path):
            with open(state_path) as f:
                _state = json.load(f)
            EXISTING_GUARDRAIL_ID = _state['guardrail_id']
            EXISTING_GUARDRAIL_VERSION = _state['guardrail_version']
            print(f"Loaded from previous run: guardrail={EXISTING_GUARDRAIL_ID}, version={EXISTING_GUARDRAIL_VERSION}")
        else:
            raise FileNotFoundError(
                f"PATH='A' but no guardrail ID provided and {state_path} not found.\n"
                "Either set EXISTING_GUARDRAIL_ID/VERSION above, or use PATH='C' to build new."
            )

    guardrail_id = EXISTING_GUARDRAIL_ID
    guardrail_version = EXISTING_GUARDRAIL_VERSION
    policies = []
    versioned_arns = []
    unique_id = "existing"

    # Load policy info from saved state
    state_path = 'data/ar_eval_state.json'
    if os.path.exists(state_path):
        with open(state_path) as f:
            _state = json.load(f)
        policies = _state.get('policies', [])
        versioned_arns = _state.get('versioned_arns', [])

    # Verify the guardrail is accessible
    resp = bedrock_runtime_client.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion=guardrail_version,
        source="OUTPUT",
        content=[{"text": {"text": "test"}}]
    )
    print(f"Using existing guardrail: {guardrail_id} (version {guardrail_version})")
    print(f"Policies loaded: {len(policies)}")
    print("Guardrail verified — skipping to Section II (Live Demo)")

elif PATH == "B":
    assert len(EXISTING_POLICY_ARNS) > 0, "PATH='B' but EXISTING_POLICY_ARNS is empty. Add your versioned policy ARNs above."
    assert len(EXISTING_POLICY_ARNS) <= 2, f"Max 2 policies per guardrail, got {len(EXISTING_POLICY_ARNS)}"
    versioned_arns = EXISTING_POLICY_ARNS
    policies = [{'policy_arn': arn.rsplit(':', 1)[0], 'versioned_arn': arn, 'status': 'COMPLETED'} for arn in versioned_arns]
    unique_id = str(round(time.time()))
    print(f"Using {len(versioned_arns)} provided policy ARN(s):")
    for arn in versioned_arns:
        print(f"  {arn}")
    print("Will create a guardrail in Section I — skipping policy build")

else:  # PATH == "C"
    print("Building new AR policies from housing code... (this takes ~10-15 minutes)")
    print("To skip this next time, set PATH = 'A' or 'B'")

ValidationException: An error occurred (ValidationException) when calling the ApplyGuardrail operation: The guardrail identifier or version provided in the request does not exist.

In [ ]:
if PATH == "C":
    RULES_PATH = 'data/housing_code_structured_rules.md'

    with open(RULES_PATH, 'r') as f:
        structured_rules = f.read()

    rule_count = structured_rules.count('RULE [')
    definition_count = structured_rules.count('DEFINITION:')
    exception_count = structured_rules.count('EXCEPTION')
    print(f"Loaded structured rules: {rule_count} rules, {definition_count} definitions, {exception_count} exceptions")
    print(f"Document size: {len(structured_rules):,} characters")
else:
    print(f"Skipped (Path {PATH})")

## I. Create an Automated Reasoning Policy

> **Path A?** These cells are skipped — you're using an existing guardrail.
> **Path B?** Policy build is skipped — we'll create a guardrail from your ARNs at the end of this section.
> **Path C?** Full build — create policies from the housing code, wait for compilation, then create the guardrail.

We create AR policies from the structured rules document. Each policy covers a domain (chapter) of the housing code. We select the top 2 domains by rule count — the maximum allowed per guardrail.

> **Key insight:** Variable descriptions are *the single most important determinant of translation accuracy*. Good descriptions include synonyms, units, boundary conditions, and plain language explanations.

In [ ]:
if PATH == "C":
    blocks = re.split(r'\n---\n', structured_rules)
    shared_defs = blocks[0].strip()

    def get_chapter_from_rule_id(rule_id):
        """Extract chapter number from rule ID like '505(a)' -> 5, '1001(a)' -> 10."""
        match = re.match(r'(\d+)', rule_id)
        if not match:
            return None
        num = int(match.group(1))
        if num >= 1000:
            return num // 100
        elif num >= 100:
            return num // 100
        else:
            return num

    chapter_blocks = {}
    for block in blocks[1:]:
        rule_ids = re.findall(r'RULE\s*\[([^\]]+)\]', block)
        if not rule_ids:
            continue
        chapters = [get_chapter_from_rule_id(rid) for rid in rule_ids]
        chapters = [c for c in chapters if c is not None]
        if not chapters:
            continue
        chapter_num = Counter(chapters).most_common(1)[0][0]
        key = f"CHAPTER_{chapter_num}"
        chapter_blocks.setdefault(key, []).append(block.strip())

    domain_docs = {}
    for chapter_key, chapter_block_list in chapter_blocks.items():
        domain_docs[chapter_key] = f"{shared_defs}\n\n---\n\n" + "\n\n---\n\n".join(chapter_block_list)

    domain_rule_counts = {
        k: sum(b.count('RULE [') for b in v)
        for k, v in chapter_blocks.items()
        if sum(b.count('RULE [') for b in v) > 0
    }
    top_domains = sorted(domain_rule_counts.items(), key=lambda x: -x[1])[:2]

    print(f"Found {len(domain_rule_counts)} domains with rules")
    print(f"\nAll domains:")
    for domain, count in sorted(domain_rule_counts.items(), key=lambda x: -x[1]):
        marker = " <--" if domain in dict(top_domains) else ""
        print(f"  {domain}: {count} rules{marker}")
    print(f"\nTop 2 domains selected for guardrail:")
    for domain, count in top_domains:
        print(f"  {domain}: {count} rules")
    if len(domain_rule_counts) > 2:
        print(f"\n  (Remaining {len(domain_rule_counts) - 2} domains available in notebook 07c)")
else:
    print(f"Skipped (Path {PATH})")

In [ ]:
if PATH == "C":
    unique_id = str(round(time.time()))
    policies = []

    # The document description is critical — it tells the INGEST_CONTENT builder
    # what to prioritize when extracting rules and variables. Include:
    # - What to ignore (editorial markers, omitted sections)
    # - Specific numeric thresholds and boundary conditions
    # - Example user questions the policy should handle
    # - Domain context and section references
    doc_description = (
        "San Francisco Housing Code regulations with if-then rules. "
        "Ignore lines containing '(omitted)' or '(portions omitted)' — "
        "these are editorial markers, not rules. Do not create rules about "
        "omitted sections. Ignore page headers, footers, editorial commentary, "
        "and lines in italics starting with underscores (e.g. '_Generated by..._'). "
        "Only extract from sections with substantive verifiable requirements. "
        "This policy validates compliance covering building construction timelines "
        "and applicable code versions (Section 104), space and occupancy standards "
        "including minimum room sizes for sleeping rooms (70 sq ft), kitchens (50 sq ft), "
        "bathrooms (30 sq ft), and per-occupant minimums (15 sq ft), plumbing and "
        "sanitary facility requirements (Section 505), community kitchen regulations "
        "(Section 507), and structural maintenance standards (Sections 601-604). "
        "Users will ask questions like 'Does this 65 sq ft bedroom meet code?', "
        "'What hot water capacity is required for a 20-unit apartment building?', "
        "'Can gas appliances be used in a community kitchen?', or "
        "'Does a building constructed in 1970 need to comply with current codes?' "
        "Priority should be given to extracting the numeric thresholds and boundary "
        "conditions for room sizes, water temperature ranges (105F-120F), hot water "
        "storage capacities, construction date cutoffs (July 26 1958, January 1 1984), "
        "and the conditional logic governing which code version applies based on "
        "building construction or alteration dates."
    )

    for domain_key, rule_count in top_domains:
        policy_name = f"housing-code-{domain_key.lower()}-{unique_id}"
        source_text = domain_docs[domain_key]

        print(f"\nCreating policy: {policy_name} ({rule_count} rules)...")

        create_resp = bedrock_client.create_automated_reasoning_policy(
            name=policy_name,
            description=f"AR policy for: {domain_key}"
        )
        policy_arn = create_resp['policyArn']
        definition_hash = create_resp['definitionHash']
        print(f"  Policy ARN: {policy_arn}")

        domain_bytes = source_text.encode('utf-8')
        build_resp = bedrock_client.start_automated_reasoning_policy_build_workflow(
            policyArn=policy_arn,
            buildWorkflowType='INGEST_CONTENT',
            sourceContent={
                'workflowContent': {
                    'documents': [{
                        'document': domain_bytes,
                        'documentContentType': 'txt',
                        'documentName': f'SF Housing Code {domain_key}',
                        'documentDescription': doc_description
                    }]
                }
            }
        )
        build_id = build_resp['buildWorkflowId']
        print(f"  Build workflow: {build_id}, waiting...")

        policies.append({
            'name': policy_name,
            'domain': domain_key,
            'policy_arn': policy_arn,
            'definition_hash': definition_hash,
            'build_workflow_id': build_id,
            'rule_count': rule_count,
            'status': 'IN_PROGRESS'
        })

    TERMINAL = {'COMPLETED', 'FAILED'}
    print(f"\n{'='*60}")
    print(f"Waiting for {len(policies)} policy builds to complete...")

    while any(p['status'] not in TERMINAL for p in policies):
        time.sleep(15)
        for p in policies:
            if p['status'] in TERMINAL:
                continue
            wf = bedrock_client.get_automated_reasoning_policy_build_workflow(
                policyArn=p['policy_arn'],
                buildWorkflowId=p['build_workflow_id']
            )
            p['status'] = wf.get('status', 'UNKNOWN')
            if p['status'] in TERMINAL:
                print(f"  {p['name']}: {p['status']}")

    print(f"\n{'='*60}")
    print(f"Build results:")
    for p in policies:
        print(f"  {p['name']}: {p['status']}")
else:
    print(f"Skipped (Path {PATH})")

In [ ]:
if PATH in ("B", "C"):
    # For Path C, version the freshly-built policies first
    if PATH == "C":
        versioned_arns = []
        for p in policies:
            if p['status'] != 'COMPLETED':
                print(f"[SKIP] {p['domain']} — status: {p['status']}")
                continue

            policy_info = bedrock_client.get_automated_reasoning_policy(policyArn=p['policy_arn'])
            p['definition_hash'] = policy_info['definitionHash']

            version_resp = bedrock_client.create_automated_reasoning_policy_version(
                policyArn=p['policy_arn'],
                lastUpdatedDefinitionHash=p['definition_hash']
            )
            policy_version = version_resp['version']
            versioned_arn = f"{p['policy_arn']}:{policy_version}"
            p['versioned_arn'] = versioned_arn
            versioned_arns.append(versioned_arn)
            print(f"Versioned: {p['name']} -> {versioned_arn}")

    # For Path B, versioned_arns was already set from EXISTING_POLICY_ARNS
    assert len(versioned_arns) > 0, f"No policy ARNs available! Check your configuration."

    guardrail_name = f"ar-eval-hub-{unique_id}"
    guardrail_resp = bedrock_client.create_guardrail(
        name=guardrail_name,
        description="AR evaluation hub guardrail — SF Housing Code",
        crossRegionConfig={
            'guardrailProfileIdentifier': 'us.guardrail.v1:0'
        },
        automatedReasoningPolicyConfig={
            'policies': versioned_arns
        },
        blockedInputMessaging="Input blocked.",
        blockedOutputsMessaging="Output blocked."
    )

    guardrail_id = guardrail_resp['guardrailId']
    guardrail_version = guardrail_resp['version']
    print(f"\nGuardrail created: {guardrail_id} (version {guardrail_version})")
else:
    print(f"Skipped (Path {PATH}) — using existing guardrail: {guardrail_id}")

In [ ]:
def get_finding_type(finding):
    """Extract the validation result and detail from a union-typed AR finding."""
    for ft in FINDING_TYPES:
        if ft in finding:
            return ft, finding[ft]
    return "unknown", {}

def has_untranslated_parts(finding_detail):
    """Check if a finding has untranslated premises or claims."""
    translation = finding_detail.get('translation', {})
    untrans_claims = translation.get('untranslatedClaims', [])
    untrans_premises = translation.get('untranslatedPremises', [])
    return len(untrans_claims) > 0 or len(untrans_premises) > 0

def get_effective_finding_type(finding):
    """Get the effective validation result, downgrading to noTranslations if there are untranslated parts.

    A 'valid' result with untranslated claims is misleading — the solver only
    checked the translated portion. We treat these as noTranslations since the
    policy couldn't fully evaluate the input.
    """
    ft, detail = get_finding_type(finding)
    if ft == 'valid' and has_untranslated_parts(detail):
        return 'noTranslations', detail
    return ft, detail

def extract_translation_metrics(finding_type, finding_detail):
    """Extract translation quality metrics from a finding's detail object."""
    if finding_type == 'translationAmbiguous':
        options = finding_detail.get('options', [])
        confidences = []
        for opt in options:
            for t in opt.get('translations', []):
                c = t.get('confidence')
                if c is not None:
                    confidences.append(c)
        return {
            'confidence': np.mean(confidences) if confidences else None,
            'premise_translation_rate': None,
            'claim_translation_rate': None,
            'has_untranslated_claims': False,
            'has_logic_warning': False,
        }

    if finding_type in ('tooComplex', 'noTranslations'):
        return {
            'confidence': None,
            'premise_translation_rate': None,
            'claim_translation_rate': None,
            'has_untranslated_claims': False,
            'has_logic_warning': False,
        }

    translation = finding_detail.get('translation', {})
    premises = translation.get('premises', [])
    claims = translation.get('claims', [])
    untranslated_premises = translation.get('untranslatedPremises', [])
    untranslated_claims = translation.get('untranslatedClaims', [])

    total_premises = len(premises) + len(untranslated_premises)
    total_claims = len(claims) + len(untranslated_claims)

    return {
        'confidence': translation.get('confidence'),
        'premise_translation_rate': len(premises) / total_premises if total_premises > 0 else 1.0,
        'claim_translation_rate': len(claims) / total_claims if total_claims > 0 else 1.0,
        'has_untranslated_claims': len(untranslated_claims) > 0,
        'has_logic_warning': 'logicWarning' in translation,
    }

def get_aggregate_finding_type(findings):
    """Aggregate multiple findings by selecting the most actionable validation result.

    Uses get_effective_finding_type to downgrade 'valid' results that have
    untranslated parts to 'noTranslations'. Then picks the finding with
    the highest priority (lowest index in FINDING_TYPES).
    """
    if not findings:
        return "noTranslations"
    best = None
    best_priority = float('inf')
    for finding in findings:
        ft, _ = get_effective_finding_type(finding)
        priority = FINDING_PRIORITY.get(ft, len(FINDING_TYPES))
        if priority < best_priority:
            best_priority = priority
            best = ft
    return best

def extract_rules_triggered(findings):
    """Collect rule IDs from findings."""
    rules = set()
    for finding in findings:
        ft, detail = get_finding_type(finding)
        for r in detail.get('supportingRules', []):
            rules.add(r.get('ruleId', 'unknown'))
        for r in detail.get('contradictingRules', []):
            rules.add(r.get('ruleId', 'unknown'))
    return list(rules)

print("Helper functions defined.")

## II. Live Demo — Validation Result Types in Action

Let's see AR Checks in action by running hand-crafted claims through our policy. Each claim targets **Chapter 5** (Space & Occupancy Standards), which our policy covers.

> **Note:** We use the **policy test API** (`start_automated_reasoning_policy_test_workflow`) rather than `apply_guardrail` for evaluation. The policy test API is the same engine that powers the "Test" button in the Bedrock console and produces accurate translations. The `apply_guardrail` API is designed for production integration but currently has a translation limitation with cross-region guardrail profiles.

In [ ]:
demo_claims = [
    {
        'label': 'SATISFIABLE — sleeping room above minimum',
        'query': 'Does an 80 square foot sleeping room meet the minimum size requirement?',
        'text': 'The sleeping room has a clear floor area of 80 square feet.',
        'expected': 'satisfiable',
        'section': '§502 (Chapter 5)'
    },
    {
        'label': 'INVALID — sleeping room below minimum',
        'query': 'Does a 50 square foot sleeping room meet the minimum size requirement?',
        'text': 'The sleeping room has a clear floor area of 50 square feet.',
        'expected': 'invalid',
        'section': '§502 (Chapter 5)'
    },
    {
        'label': 'SATISFIABLE — showerhead compliant flow rate',
        'query': 'Does a showerhead with a flow rate of 2.5 gallons per minute comply with the code?',
        'text': 'The showerhead has a flow rate of 2.5 gallons per minute.',
        'expected': 'satisfiable',
        'section': '§505(d)(5) (Chapter 5)'
    },
    {
        'label': 'SATISFIABLE — hot water in range',
        'query': 'Does hot water at 110 degrees Fahrenheit at the tap comply with the housing code?',
        'text': 'The apartment building maintains hot water at the tap at 110 degrees Fahrenheit.',
        'expected': 'satisfiable',
        'section': '§505(d)(3) (Chapter 5)'
    },
    {
        'label': 'INVALID — kitchen below minimum',
        'query': 'Does a 40 square foot kitchen meet the housing code requirements?',
        'text': 'The kitchen has a clear floor area of 40 square feet.',
        'expected': 'invalid',
        'section': '§502 (Chapter 5)'
    },
    {
        'label': 'NO_TRANSLATION — off-topic',
        'query': 'Where can I find the best pizza in San Francisco?',
        'text': "The best pizza in San Francisco can be found at Tony's in North Beach.",
        'expected': 'noTranslations',
        'section': 'N/A'
    },
]

# We use the policy test API for demo because apply_guardrail currently has a
# translation issue with cross-region guardrails. The policy test API uses the
# same translation engine that powers the Bedrock console's "Test" button.

print("Running live demo claims through AR policy test API...\n")

ch5_arn = policies[0]['policy_arn'] if policies else state['policies'][0]['policy_arn'] if 'state' not in dir() else None
if ch5_arn is None:
    import json as _json
    with open('data/ar_eval_state.json') as _f:
        _state = _json.load(_f)
    ch5_arn = _state['policies'][0]['policy_arn']

# Get the build workflow ID
_wf_resp = bedrock_client.list_automated_reasoning_policy_build_workflows(
    policyArn=ch5_arn, maxResults=5)
build_workflow_id = None
for _wf in _wf_resp['automatedReasoningPolicyBuildWorkflowSummaries']:
    if _wf['buildWorkflowType'] == 'INGEST_CONTENT' and _wf['status'] == 'COMPLETED':
        build_workflow_id = _wf['buildWorkflowId']
        break

for claim in demo_claims:
    # Create a temporary test case
    tc_kwargs = {
        'policyArn': ch5_arn,
        'guardContent': claim['text'],
        'expectedAggregatedFindingsResult': 'VALID',
        'confidenceThreshold': 0.0,
    }
    if claim.get('query'):
        tc_kwargs['queryContent'] = claim['query']
    tc_resp = bedrock_client.create_automated_reasoning_policy_test_case(**tc_kwargs)
    tc_id = tc_resp['testCaseId']

    # Run the test
    bedrock_client.start_automated_reasoning_policy_test_workflow(
        policyArn=ch5_arn,
        buildWorkflowId=build_workflow_id,
        testCaseIds=[tc_id]
    )

    # Poll for result
    actual = '?'
    confidence = None
    for _ in range(15):
        time.sleep(3)
        res = bedrock_client.list_automated_reasoning_policy_test_results(
            policyArn=ch5_arn, buildWorkflowId=build_workflow_id, maxResults=100)
        for tr in res.get('testResults', []):
            if tr['testCase']['testCaseId'] == tc_id and tr.get('testRunStatus') == 'COMPLETED':
                actual = tr.get('aggregatedTestFindingsResult', '?')
                for f in tr.get('testFindings', []):
                    raw = f.get('raw', {})
                    conf = raw.get('translation', {}).get('confidence')
                    if conf is not None:
                        confidence = conf
                break
        if actual != '?':
            break

    # Normalize validation result names
    verdict_norm = {
        'VALID': 'valid', 'INVALID': 'invalid', 'SATISFIABLE': 'satisfiable',
        'NO_TRANSLATION': 'noTranslations', 'NO_TRANSLATIONS': 'noTranslations',
        'TRANSLATION_AMBIGUOUS': 'translationAmbiguous',
    }
    ft = verdict_norm.get(actual, actual)
    conf_str = f" (confidence: {confidence:.2f})" if confidence else ""
    match = "MATCH" if ft == claim['expected'] else "MISMATCH"

    print(f"{'='*70}")
    print(f"  {claim['label']}  [{claim['section']}]")
    print(f"  Claim: \"{claim['text'][:80]}\"")
    print(f"  Validation Result: {ft}{conf_str}  [{match}]")
    print()

    # Cleanup test case
    tc_info = bedrock_client.get_automated_reasoning_policy_test_case(
        policyArn=ch5_arn, testCaseId=tc_id)
    bedrock_client.delete_automated_reasoning_policy_test_case(
        policyArn=ch5_arn, testCaseId=tc_id, lastUpdatedAt=tc_info['testCase']['updatedAt'])

## III. Metrics Framework — What to Measure and Why

Before running the evaluation, let's define **what we're measuring** and **why each metric matters** for AR policy quality.

### The 5 Key Metrics

| Metric | Question It Answers | Why It Matters |
|--------|-------------------|----------------|
| **False Valid Rate** | Is it safe? | The most critical metric. A false valid means the policy certified a non-compliant claim as compliant — the AR equivalent of a safety system saying "all clear" when there's a real problem. **Target: 0%** |
| **Consistency Accuracy** | Is it predictable? | Measures how often the policy produces the validation result we expect. A high score means the policy behaves reliably across runs. **Target: >80%** |
| **Macro F1** | Is it balanced? | Average F1 across all validation result types, weighted equally. Catches cases where accuracy looks high but one validation result type is always wrong (e.g., never detecting INVALID). **Target: >0.8** |
| **Ideal Accuracy** | Is it good enough? | Measures how often the actual validation result matches the *ideal* answer — what the validation result should be with perfect translation. Low ideal accuracy means the policy needs variable description improvements. |
| **Fidelity Gap** | How much can we improve? | The difference between consistency accuracy and ideal accuracy. A large gap means the policy is *predictable but weak* — it consistently gives the same answer, but that answer could be better. |

### How They Fit Together

```
Step 1: False Valid Rate = 0%       → The policy is SAFE (won't certify bad claims)
Step 2: Accuracy > 80%              → The policy is PREDICTABLE (does what we expect)
Step 3: Macro F1 > 0.8              → The policy is BALANCED (works across all validation result types)
Step 4: Close the Fidelity Gap      → The policy is GOOD ENOUGH (ideal matches actual)
```

### Two Levels of Expectations

Each test case has two expectation fields:

- **`expected_finding_type`** — what the policy *actually produces* with the current variable descriptions. This is what we measure consistency against.
- **`ideal_finding_type`** — what the validation result *should be* based on the source document. The gap between these two measures **translation fidelity** — how well the policy's variables capture the rules.

### Per-Type Precision, Recall, and F1

We compute P/R/F1 for each validation result type separately:

| Metric | What It Catches |
|--------|----------------|
| **Precision** (P) | Of all times we predicted this validation result, how many were correct? Low precision = too many false alarms. |
| **Recall** (R) | Of all tests that should be this validation result, how many did we catch? Low recall = missing cases. |
| **F1** | Harmonic mean of P and R. Balances false alarms vs missed cases. |

For example, `invalid` P=1.00 R=0.88 means: every time the policy says INVALID it's correct (no false alarms), but it misses 12% of truly invalid claims (some slip through as SATISFIABLE or IMPOSSIBLE).

## IV. Run the Evaluation

Now let's run 35 test cases through the policy test API and compute the metrics defined above.

**How it works:** For each test, we create a temporary test case on the appropriate policy (using both the question and the LLM answer), run the test workflow, collect the validation result, then clean up.

**Test set composition:**

| Category | Count | Purpose |
|----------|-------|---------|
| **validation_correctness** | 25 | Known-compliant and non-compliant claims across sleeping rooms, kitchens, bathrooms, plumbing, and more |
| **boundary_value** | 2 | Numeric thresholds at exact boundary |
| **adversarial_negation** | 1 | Negated rule — validation result must flip |
| **translation_quality** | 1 | Colloquial language, abbreviations |
| **coverage_probe** | 3 | Out-of-scope claims (expected: `noTranslations`) |

In [ ]:
with open('data/ar_tests_essential.json') as f:
    test_cases = json.load(f)

df_tests = pd.DataFrame(test_cases)
print(f"Loaded {len(test_cases)} essential test cases\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_tests['test_category'].value_counts().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Tests by Category')
axes[0].set_xlabel('Count')
df_tests['expected_finding_type'].value_counts().plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Tests by Expected Validation Result')
axes[1].set_xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
def get_build_workflow_id(policy_arn):
    """Find the completed INGEST_CONTENT build workflow for a policy."""
    resp = bedrock_client.list_automated_reasoning_policy_build_workflows(
        policyArn=policy_arn, maxResults=5)
    for wf in resp['automatedReasoningPolicyBuildWorkflowSummaries']:
        if wf['buildWorkflowType'] == 'INGEST_CONTENT' and wf['status'] == 'COMPLETED':
            return wf['buildWorkflowId']
    return None

def get_chapter_from_section(section):
    """Map section number to chapter: '502' → 5, '1001' → 10."""
    match = re.match(r'(\d+)', section)
    if not match:
        return None
    num = int(match.group(1))
    return num // 100 if num >= 100 else num

def run_ar_evaluation(test_cases, policies):
    """Run test cases through the AR policy test API and collect results."""
    # Build policy lookup: chapter → (arn, build_workflow_id)
    policy_lookup = {}
    for p in policies:
        domain = p.get('domain', '')
        match = re.search(r'(\d+)', domain)
        if match:
            ch = int(match.group(1))
            build_id = get_build_workflow_id(p['policy_arn'])
            policy_lookup[ch] = (p['policy_arn'], build_id)

    # Default to the first policy for out-of-scope tests
    default_arn, default_build = list(policy_lookup.values())[0]

    results = []
    total = len(test_cases)

    for i, test in enumerate(test_cases):
        if (i + 1) % 10 == 0 or i == 0:
            print(f"Processing test {i+1}/{total}...")

        ch = get_chapter_from_section(test.get('housing_code_section', ''))
        policy_arn, build_id = policy_lookup.get(ch, (default_arn, default_build))

        result = {
            'test_number': test['test_number'],
            'test_category': test['test_category'],
            'housing_code_section': test.get('housing_code_section', ''),
            'rule_domain': test.get('rule_domain', ''),
            'expected_finding_type': test['expected_finding_type'],
            'ideal_finding_type': test.get('ideal_finding_type', test['expected_finding_type']),
            'difficulty': test.get('difficulty', ''),
            'input_length': len(test['llm_output']),
        }

        try:
            # Create temporary test case
            tc_kwargs = {
                'policyArn': policy_arn,
                'guardContent': test['llm_output'],
                'expectedAggregatedFindingsResult': 'VALID',
                'confidenceThreshold': 0.0,
            }
            if test.get('query'):
                tc_kwargs['queryContent'] = test['query']
            tc_resp = bedrock_client.create_automated_reasoning_policy_test_case(**tc_kwargs)
            tc_id = tc_resp['testCaseId']

            # Run test workflow
            start_time = time.time()
            bedrock_client.start_automated_reasoning_policy_test_workflow(
                policyArn=policy_arn,
                buildWorkflowId=build_id,
                testCaseIds=[tc_id]
            )

            # Poll for result
            actual_raw = '?'
            findings_data = []
            for _ in range(20):
                time.sleep(3)
                res = bedrock_client.list_automated_reasoning_policy_test_results(
                    policyArn=policy_arn, buildWorkflowId=build_id, maxResults=100)
                for tr in res.get('testResults', []):
                    if tr['testCase']['testCaseId'] == tc_id and tr.get('testRunStatus') == 'COMPLETED':
                        actual_raw = tr.get('aggregatedTestFindingsResult', '?')
                        findings_data = tr.get('testFindings', [])
                        break
                if actual_raw != '?':
                    break

            latency_ms = (time.time() - start_time) * 1000

            # Normalize validation result
            verdict_norm = {
                'VALID': 'valid', 'INVALID': 'invalid', 'SATISFIABLE': 'satisfiable',
                'IMPOSSIBLE': 'impossible', 'NO_TRANSLATION': 'noTranslations',
                'NO_TRANSLATIONS': 'noTranslations', 'TOO_COMPLEX': 'tooComplex',
                'TRANSLATION_AMBIGUOUS': 'translationAmbiguous',
            }
            actual_type = verdict_norm.get(actual_raw, actual_raw)

            # Extract metrics from findings
            confidences = []
            rules_triggered = set()
            has_untranslated = False
            for f in findings_data:
                raw = f.get('raw', {})
                translation = raw.get('translation', {})
                conf = translation.get('confidence')
                if conf is not None:
                    confidences.append(conf)
                if translation.get('untranslatedClaims'):
                    has_untranslated = True
                for r in raw.get('supportingRules', []):
                    rules_triggered.add(r.get('id', 'unknown'))
                for r in raw.get('contradictingRules', []):
                    rules_triggered.add(r.get('id', 'unknown'))

            result['actual_finding_type'] = actual_type
            result['finding_correct'] = (actual_type == test['expected_finding_type'])
            result['ideal_correct'] = (actual_type == result['ideal_finding_type'])
            result['num_findings'] = len(findings_data)
            result['latency_ms'] = latency_ms
            result['rules_triggered'] = list(rules_triggered)
            result['api_error'] = None
            result['avg_confidence'] = np.mean(confidences) if confidences else None
            result['has_untranslated_claims'] = has_untranslated

            # Cleanup test case
            tc_info = bedrock_client.get_automated_reasoning_policy_test_case(
                policyArn=policy_arn, testCaseId=tc_id)
            bedrock_client.delete_automated_reasoning_policy_test_case(
                policyArn=policy_arn, testCaseId=tc_id,
                lastUpdatedAt=tc_info['testCase']['updatedAt'])

        except Exception as e:
            result['actual_finding_type'] = 'error'
            result['finding_correct'] = False
            result['ideal_correct'] = False
            result['api_error'] = str(e)
            result['latency_ms'] = None
            result['avg_confidence'] = None
            result['has_untranslated_claims'] = False

        results.append(result)

    return results

# Load policy info
if not policies:
    with open('data/ar_eval_state.json') as f:
        _state = json.load(f)
    policies = _state['policies']

print("Starting AR evaluation via policy test API...\n")
results = run_ar_evaluation(test_cases, policies)
df = pd.DataFrame(results)

accuracy = df['finding_correct'].mean()
api_errors = df['api_error'].notna().sum()
print(f"\n{'='*60}")
print(f"Evaluation complete: {len(results)} tests")
print(f"Overall accuracy: {accuracy:.1%}")
if 'ideal_correct' in df.columns:
    ideal_acc = df['ideal_correct'].mean()
    print(f"Ideal accuracy:  {ideal_acc:.1%}")
print(f"API errors: {api_errors}")

In [ ]:
def compute_core_metrics(df, expected_col='expected_finding_type'):
    """Compute essential metrics for the hub evaluation."""
    valid_df = df[df['api_error'].isna()]
    accuracy = valid_df['finding_correct'].mean() if 'finding_correct' in valid_df else (valid_df['actual_finding_type'] == valid_df[expected_col]).mean()

    labels = [ft for ft in FINDING_TYPES
              if ft in valid_df[expected_col].values
              or ft in valid_df['actual_finding_type'].values]

    cm = pd.crosstab(valid_df[expected_col], valid_df['actual_finding_type'],
                      dropna=False).reindex(index=labels, columns=labels, fill_value=0)

    per_type = {}
    for ft in labels:
        tp = cm.loc[ft, ft] if ft in cm.index and ft in cm.columns else 0
        fp = cm[ft].sum() - tp if ft in cm.columns else 0
        fn = cm.loc[ft].sum() - tp if ft in cm.index else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_type[ft] = {'precision': precision, 'recall': recall, 'f1': f1}

    types_with_data = [ft for ft in labels if ft in cm.index and cm.loc[ft].sum() > 0]
    macro_f1 = np.mean([per_type[ft]['f1'] for ft in types_with_data]) if types_with_data else 0

    expected_non_valid = valid_df[valid_df[expected_col] != 'valid']
    false_valid = expected_non_valid[expected_non_valid['actual_finding_type'] == 'valid']
    false_valid_rate = len(false_valid) / len(expected_non_valid) if len(expected_non_valid) > 0 else 0

    confidences = valid_df['avg_confidence'].dropna()
    mean_confidence = confidences.mean() if len(confidences) > 0 else None

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'false_valid_rate': false_valid_rate,
        'false_valid_count': len(false_valid),
        'mean_confidence': mean_confidence,
        'confusion_matrix': cm,
        'per_type': per_type,
        'n_evaluated': len(valid_df),
    }

# Consistency metrics: actual vs expected (what the policy should produce)
metrics = compute_core_metrics(df, 'expected_finding_type')

print(f"{'='*60}")
print(f"  Consistency Metrics (actual vs. expected)")
print(f"{'='*60}")
print(f"  Accuracy:            {metrics['accuracy']:.1%}")
print(f"  Macro F1:            {metrics['macro_f1']:.3f}")
print(f"  Mean Confidence:     {metrics['mean_confidence']:.3f}" if metrics['mean_confidence'] else "  Mean Confidence:     N/A")
print()

# Ideal metrics: actual vs ideal (the correct answer with perfect translation)
if 'ideal_finding_type' in df.columns:
    ideal_accuracy = (df[df['api_error'].isna()]['actual_finding_type'] == df[df['api_error'].isna()]['ideal_finding_type']).mean()
    print(f"{'='*60}")
    print(f"  Translation Fidelity (actual vs. ideal)")
    print(f"{'='*60}")
    print(f"  Ideal Accuracy:      {ideal_accuracy:.1%}")
    print(f"  Fidelity Gap:        {metrics['accuracy'] - ideal_accuracy:+.1%}")
    print(f"  (The gap measures how much optimization could improve results)")
    print()

if metrics['false_valid_rate'] > 0:
    print(f"  *** FALSE VALID RATE: {metrics['false_valid_rate']:.1%} ({metrics['false_valid_count']} cases) ***")
    print(f"  *** This is the most dangerous failure mode — invalid claims certified as valid ***")
else:
    print(f"  False VALID Rate:    {metrics['false_valid_rate']:.1%} (target: 0%)")
print()
print(f"  Per-Type Performance:")
for ft, m in metrics['per_type'].items():
    print(f"    {ft:25s}  P={m['precision']:.2f}  R={m['recall']:.2f}  F1={m['f1']:.2f}")

# Summary: what each metric tells you
fidelity_gap = metrics['accuracy'] - ideal_accuracy if 'ideal_accuracy' in dir() else None
print(f"\n{'='*60}")
print(f"  What It All Means")
print(f"{'='*60}")
print(f"  False Valid Rate   {metrics['false_valid_rate']:>6.1%}  Is it safe?           {'Yes' if metrics['false_valid_rate'] == 0 else 'NO'}")
print(f"  Accuracy           {metrics['accuracy']:>6.1%}  Is it predictable?    {'Yes' if metrics['accuracy'] >= 0.8 else 'No'}")
print(f"  Macro F1           {metrics['macro_f1']:>6.3f}  Is it balanced?       {'Yes' if metrics['macro_f1'] >= 0.8 else 'No'}")
if fidelity_gap is not None:
    print(f"  Ideal Accuracy     {ideal_accuracy:>6.1%}  Is it good enough?    {'Yes' if ideal_accuracy >= 0.8 else 'Not yet'}")
    print(f"  Fidelity Gap       {fidelity_gap:>+6.1%}  How much to improve?  {'Low' if abs(fidelity_gap) < 0.2 else 'Significant'}")

## V. Visualizations

In [ ]:
cm = metrics['confusion_matrix']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=cm.columns, yticklabels=cm.index)
ax.set_xlabel('Actual Validation Result')
ax.set_ylabel('Expected Validation Result')
ax.set_title('AR Confusion Matrix \u2014 Expected vs. Actual Validation Results')
plt.tight_layout()
plt.show()

In [ ]:
categories_radar = ['Accuracy', 'Macro F1', 'Confidence', 'No False Valid']
values = [
    metrics['accuracy'],
    metrics['macro_f1'],
    metrics['mean_confidence'] if metrics['mean_confidence'] else 0,
    1.0 - metrics['false_valid_rate'],
]
target = 0.8

angles = np.linspace(0, 2 * np.pi, len(categories_radar), endpoint=False).tolist()
values_plot = values + [values[0]]
angles_plot = angles + [angles[0]]
target_plot = [target] * (len(categories_radar) + 1)

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles_plot, values_plot, 'o-', linewidth=2, label='Actual')
ax.fill(angles_plot, values_plot, alpha=0.25)
ax.plot(angles_plot, target_plot, '--', color='red', linewidth=1, alpha=0.5, label='Target (0.8)')
ax.set_xticks(angles)
ax.set_xticklabels(categories_radar)
ax.set_ylim(0, 1.0)
ax.set_title('Evaluation Dimensions')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

In [ ]:
pt = metrics['per_type']
types = [ft for ft in pt if pt[ft]['f1'] > 0 or ft in df['expected_finding_type'].values]
x = np.arange(len(types))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, [pt[ft]['precision'] for ft in types], width, label='Precision', color='steelblue')
ax.bar(x, [pt[ft]['recall'] for ft in types], width, label='Recall', color='coral')
ax.bar(x + width, [pt[ft]['f1'] for ft in types], width, label='F1', color='forestgreen')
ax.set_xticks(x)
ax.set_xticklabels(types, rotation=45, ha='right')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 by Validation Result')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
cat_acc = df.groupby('test_category')['finding_correct'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#d32f2f' if v < 0.6 else '#f9a825' if v < 0.8 else '#388e3c' for v in cat_acc]
cat_acc.plot.barh(ax=ax, color=colors)
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5, label='Target (80%)')
ax.set_xlabel('Accuracy')
ax.set_title('Accuracy by Test Category')
ax.legend()
plt.tight_layout()
plt.show()

## VI. Investigation — Why Do Some Tests Mismatch?

Our evaluation hit ~91-100% accuracy depending on the run. When mismatches occur, they aren't bugs — they reveal how **question context changes the AR translator's behavior**. This is one of the most important things to understand about AR Checks.

Each test case has two parts:
- **Query** (the user's question): provides context about what's being asked
- **Guard content** (the LLM's answer): the claim being verified against the policy

The AR translator uses *both* to decide what counts as a **premise** (assumed fact) vs a **claim** (assertion to verify). When the question establishes a fact ("Does a *hotel* providing hot water at 130°F..."), the translator may treat "hotel" as a premise rather than a claim — which changes the validation result.

### Exercise: Investigate the mismatches

Run the cell below to see exactly what happened for each mismatch: what we expected, what the policy returned, and why.

In [ ]:
# Find and display all mismatches with context
mismatches = df[~df['finding_correct']].copy()

if len(mismatches) == 0:
    print("No mismatches found — all tests matched expectations!")
else:
    print(f"Found {len(mismatches)} mismatches to investigate:\n")
    
    for _, row in mismatches.iterrows():
        test = next(t for t in test_cases if t['test_number'] == row['test_number'])
        
        print(f"{'='*70}")
        print(f"  Test #{row['test_number']} — {row['test_category']} ({row['housing_code_section']})")
        print(f"{'='*70}")
        print(f"  Question:  \"{test.get('query', 'N/A')}\"")
        print(f"  Answer:    \"{test['llm_output']}\"")
        print(f"  Expected:  {row['expected_finding_type']}")
        print(f"  Actual:    {row['actual_finding_type']}")
        print(f"  Findings:  {row['num_findings']}")
        print()
        
        # Explain why this might have happened
        if row['actual_finding_type'] == 'impossible':
            print(f"  WHY: The translator found contradictory premises. The question")
            print(f"  likely established a fact that contradicts the answer. For example,")
            print(f"  asking 'does X meet the requirement?' while the answer states a")
            print(f"  value below the minimum creates a logical contradiction.")
        elif row['actual_finding_type'] == 'invalid' and row['expected_finding_type'] == 'satisfiable':
            print(f"  WHY: The question provided enough context for the translator to")
            print(f"  establish premises that turned SATISFIABLE into INVALID. Without")
            print(f"  the question, the solver couldn't determine the structure type;")
            print(f"  with it, the structure type became a premise, and the value")
            print(f"  violated a rule.")
        print()

### Deep dive: How does the question change the validation result?

Pick one of the mismatched tests above and run it **with and without** the question. This shows how the same answer gets a different validation result depending on context.

Modify `TEST_NUMBER` below to investigate a specific test.

In [ ]:
# ── Change this to investigate a specific test ──────────────
TEST_NUMBER = 15  # Try 15, 17, or 28

test = next(t for t in test_cases if t['test_number'] == TEST_NUMBER)
ch5_arn = policies[0]['policy_arn']
build_wf_id = get_build_workflow_id(ch5_arn)

verdict_norm = {
    'VALID': 'valid', 'INVALID': 'invalid', 'SATISFIABLE': 'satisfiable',
    'IMPOSSIBLE': 'impossible', 'NO_TRANSLATION': 'noTranslations',
    'TRANSLATION_AMBIGUOUS': 'translationAmbiguous',
}

def run_single_test(policy_arn, build_id, guard_content, query_content=None):
    """Run a single test case and return the validation result with translation details."""
    tc_kwargs = {
        'policyArn': policy_arn,
        'guardContent': guard_content,
        'expectedAggregatedFindingsResult': 'VALID',
        'confidenceThreshold': 0.0,
    }
    if query_content:
        tc_kwargs['queryContent'] = query_content
    tc = bedrock_client.create_automated_reasoning_policy_test_case(**tc_kwargs)
    tc_id = tc['testCaseId']

    bedrock_client.start_automated_reasoning_policy_test_workflow(
        policyArn=policy_arn, buildWorkflowId=build_id, testCaseIds=[tc_id])

    for _ in range(20):
        time.sleep(3)
        res = bedrock_client.list_automated_reasoning_policy_test_results(
            policyArn=policy_arn, buildWorkflowId=build_id, maxResults=100)
        for tr in res.get('testResults', []):
            if tr['testCase']['testCaseId'] == tc_id and tr.get('testRunStatus') == 'COMPLETED':
                verdict = verdict_norm.get(tr.get('aggregatedTestFindingsResult', '?'), '?')
                findings = tr.get('testFindings', [])

                # Extract translation details from first finding
                premises, claims, untranslated = [], [], []
                for f in findings:
                    raw = f.get('raw', {})
                    t_data = raw.get('translation', {})
                    for p in t_data.get('premises', []):
                        premises.append(p.get('naturalLanguage', ''))
                    for c in t_data.get('claims', []):
                        claims.append(c.get('naturalLanguage', ''))
                    for u in t_data.get('untranslatedClaims', []):
                        untranslated.append(u.get('text', str(u)))

                # Cleanup
                tc_info = bedrock_client.get_automated_reasoning_policy_test_case(
                    policyArn=policy_arn, testCaseId=tc_id)
                bedrock_client.delete_automated_reasoning_policy_test_case(
                    policyArn=policy_arn, testCaseId=tc_id,
                    lastUpdatedAt=tc_info['testCase']['updatedAt'])

                return {
                    'verdict': verdict,
                    'premises': premises,
                    'claims': claims,
                    'untranslated': untranslated,
                    'num_findings': len(findings),
                }
    return {'verdict': 'timeout', 'premises': [], 'claims': [], 'untranslated': [], 'num_findings': 0}


print(f"Test #{TEST_NUMBER}")
print(f"  Question: \"{test.get('query', 'N/A')}\"")
print(f"  Answer:   \"{test['llm_output']}\"")
print(f"  Expected: {test['expected_finding_type']}")

# Run WITHOUT query
print(f"\n{'─'*60}")
print(f"  Run 1: Answer ONLY (no question)")
print(f"{'─'*60}")
r1 = run_single_test(ch5_arn, build_wf_id, test['llm_output'])
print(f"  Validation Result:    {r1['verdict']}")
print(f"  Premises:   {r1['premises'] or '(none)'}")
print(f"  Claims:     {r1['claims'] or '(none)'}")
if r1['untranslated']:
    print(f"  Untranslated: {r1['untranslated']}")

# Run WITH query
print(f"\n{'─'*60}")
print(f"  Run 2: Question + Answer")
print(f"{'─'*60}")
r2 = run_single_test(ch5_arn, build_wf_id, test['llm_output'], test.get('query'))
print(f"  Validation Result:    {r2['verdict']}")
print(f"  Premises:   {r2['premises'] or '(none)'}")
print(f"  Claims:     {r2['claims'] or '(none)'}")
if r2['untranslated']:
    print(f"  Untranslated: {r2['untranslated']}")

# Summary
print(f"\n{'─'*60}")
if r1['verdict'] != r2['verdict']:
    print(f"  RESULT CHANGED: {r1['verdict']} → {r2['verdict']}")
    print(f"  The question provided additional context that changed how the")
    print(f"  translator split the input into premises vs claims.")
else:
    print(f"  Result unchanged: {r1['verdict']} (question had no effect)")

### Key takeaways

1. **Questions act as premises.** When the question says "Does a *hotel* providing hot water at 130°F...", the translator treats `structureType = HOTEL` as an established fact. This narrows the search space and can flip a SATISFIABLE validation result to INVALID (or vice versa).

2. **IMPOSSIBLE means contradictory premises.** If the question implies one thing and the answer states another, the solver detects the contradiction. This is the AR system working correctly — it found a logical inconsistency in the input.

3. **Context sensitivity is a feature, not a bug.** In a real application, the user's question provides context that the LLM's answer alone may not. AR Checks are more accurate with both Q and A than with A alone.

### What to try next

- **Adjust the question** for a mismatched test to see if you can get the expected validation result. What phrasing gives the translator enough context without creating contradictions?
- **Look at the translation details** (premises vs claims) to understand exactly what the translator extracted. Are there premises you didn't expect?
- **Improve variable descriptions** in the policy to help the translator make better translations. This is covered in detail in notebook `04-06-04`.

## VII. Save State & Next Steps

Saving evaluation state so spoke notebooks can resume without re-creating policies.

In [ ]:
checkpoint = {
    'guardrail_id': guardrail_id,
    'guardrail_version': guardrail_version,
    'policies': policies,
    'versioned_arns': versioned_arns,
    'unique_id': unique_id,
    'test_results': results,
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}

with open('data/ar_eval_state.json', 'w') as f:
    json.dump(checkpoint, f, indent=2, default=str)

print("Checkpoint saved to data/ar_eval_state.json")
print(f"  Guardrail: {guardrail_id} (version {guardrail_version})")
print(f"  Policies: {len(policies)}")
print(f"  Test results: {len(results)}")

## Bring Your Own: Applying This to Your Data

This notebook used the SF Housing Code as an example, but the evaluation framework works with **any AR policy and any domain**. Here's how to adapt it to your own use case.

### Step 1: Prepare your source document

AR policies are built from regulatory documents, compliance manuals, internal policies, or any document with verifiable rules. Supported formats: PDF, TXT, or structured markdown.

**Tips for good source documents:**
- Include explicit thresholds and boundary conditions (e.g., "minimum 70 square feet", "no more than 3 gallons per minute")
- Use clear if-then language where possible
- Break large documents into domain-specific chunks (the AR builder has a 2-policy-per-guardrail limit)

### Step 2: Build your AR policy

Use **Path C** in this notebook as a template. Replace `data/housing_code_structured_rules.md` with your document and update the `doc_description` in cell `cell-08-create-policies` to describe your domain.

### Step 3: Create your test suite

Create a JSON file following the same schema as `data/ar_tests_essential.json`:

```json
[
  {
    "test_number": 1,
    "test_category": "validation_correctness",
    "housing_code_section": "your-section-ref",
    "rule_domain": "your-domain",
    "difficulty": "easy",
    "query": "Does [specific scenario] comply with [your policy]?",
    "llm_output": "Yes, [specific factual assertion matching policy variables]. The [value] meets the [threshold].",
    "expected_finding_type": "valid",
    "ideal_finding_type": "valid"
  }
]
```

**Test design tips:**
- **For VALID tests:** The LLM answer should state premises (room type, structure type) as facts AND reference the compliance conclusion using language close to the policy's variable names
- **For INVALID tests:** State values that clearly violate a rule threshold
- **For SATISFIABLE tests:** State facts without enough context for the solver to reach a definitive conclusion
- **For NO_TRANSLATIONS tests:** Use claims completely outside your policy's domain
- **Include a mix of difficulties:** Easy (clear pass/fail), medium (boundary values), hard (negations, abbreviations, colloquial language)

### Step 4: Calibrate expectations

Your first run will likely show mismatches. This is expected! Use the investigation cells in Section VI to understand why, then update `expected_finding_type` to match what the policy actually produces. Keep `ideal_finding_type` as the ground truth from your source document.

### Step 5: Iterate on the policy

The **Fidelity Gap** (consistency accuracy - ideal accuracy) tells you how much room there is to improve. To close the gap:

1. **Inspect policy variables** in the Bedrock console — are the variable descriptions clear enough for the translator?
2. **Add synonyms and units** to variable descriptions — e.g., "square feet", "sq ft", "sqft" should all map to the same variable
3. **Improve Q/A phrasing** — questions that establish context as premises help the translator distinguish facts from claims
4. **Rebuild the policy** after updating variable descriptions and re-run the evaluation to measure improvement

### Step 6: Monitor in production

Once your Fidelity Gap is small enough:
- Track the **False Valid Rate** continuously — any non-zero value needs immediate investigation
- Re-run the test suite after every policy rebuild to catch regressions
- Add new test cases as you discover edge cases in production

In [ ]:
SKIP_CLEANUP = True  # Set to False to delete resources

if SKIP_CLEANUP:
    print("Skipping cleanup \u2014 resources preserved for spoke notebooks.")
    print("Set SKIP_CLEANUP = False and re-run this cell to delete.")
else:
    try:
        bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_id)
        print(f"Deleted guardrail: {guardrail_id}")
    except Exception as e:
        print(f"Error deleting guardrail: {e}")

    for p in policies:
        try:
            bedrock_client.delete_automated_reasoning_policy(
                policyArn=p['policy_arn']
            )
            print(f"Deleted policy: {p['name']}")
        except Exception as e:
            print(f"Error deleting policy {p['name']}: {e}")

    print("\nCleanup complete.")